# Exercice 5 : Contourner l'IDS

> **En une phrase :** l'attaquant modifie une attaque détectée juste assez pour qu'elle passe, en se servant de sa copie pour viser.

**Le scénario.** L'attaquant a une attaque que l'IDS détecte. Il veut la modifier **un peu** (sans changer ce qu'elle fait) pour qu'elle soit classée « normal ».
Il ne veut pas envoyer des milliers d'essais à l'IDS : il se ferait repérer. Il s'entraîne donc sur ses copies.

**Vocabulaire de l'exercice.**
| Mot | Sens |
|---|---|
| **copie arbre** | un arbre de décision copié sur l'IDS (ex. 2). Lisible : il dit **où viser**. |
| **copie forêt** | une forêt copiée sur l'IDS. Plus fidèle : elle sert à **tester** les variantes gratuitement. |
| **recette** | les plages de valeurs à viser pour que l'arbre dise « normal ». |
| **variante** | l'attaque avec quelques features modifiées. |
| **effort** | à quel point la variante est modifiée (0 = identique). Plus c'est petit, plus c'est discret. |

**Le plan.**
1. Préparer l'IDS et voler les deux copies.
2. Choisir une attaque et lire sa recette dans l'arbre.
3. Générer des variantes : à l'aveugle, puis guidé par la recette.
4. Recommencer sur 60 attaques.
5. Le raccourci : et si l'attaquant touche au TTL ?
6. Contre-mesure : retirer le raccourci.

In [ ]:
# Sur Google Colab seulement : récupère helper.py et les données (en local, cette cellule ne fait rien)
import os, sys
if 'google.colab' in sys.modules and not os.path.exists('helper.py'):
    !git clone -q --depth 1 https://github.com/elyesmanai/8IAR805---Cybersecurity-AI.git /content/cours
    %cd "/content/cours/Chapitre 5 - Sécurité de l'IA"

## Étape 1 : Préparer l'IDS et voler les deux copies
**Ce qu'on fait :** l'attaquant envoie 10 000 requêtes à l'API (comme à l'ex. 2) et entraîne ses deux copies.

In [1]:
from helper import preparer
X_train, y_train, X_test, y_test, types_attaque, ids = preparer()

from helper import API, voler_copies
api = API(ids)
arbre, foret = voler_copies(api, X_test, n_requetes=10000)

trafic_nouveau = X_train.sample(5000, random_state=2)
print(f"Fidélité de l'arbre : {(arbre.predict(trafic_nouveau) == ids.predict(trafic_nouveau)).mean():.0%}")
print(f"Fidélité de la forêt : {(foret.predict(trafic_nouveau) == ids.predict(trafic_nouveau)).mean():.0%}")

IDS de l'entreprise -> accuracy = 0.901 | recall (attaques détectées) = 0.864


Fidélité de l'arbre : 93%
Fidélité de la forêt : 94%


**👀 Ce qu'on doit voir :** les deux copies sont fidèles à ≈ 93-94 %.

## Étape 2 : Choisir une attaque et lire sa recette
**Ce qu'on fait :** on prend une attaque détectée de justesse (dans la zone grise de l'ex. 1). On cherche dans l'arbre les chemins qui mènent à « normal » en ne touchant **que** les features contrôlables.

In [2]:
import pandas as pd
from helper import recettes_substitut, recette_en_tableau

confiance = pd.Series(ids.predict_proba(X_test)[:, 1], index=X_test.index)

# <-- MODIFIABLE : une autre attaque, par exemple X_test.loc[[51042]]
cible = X_test.loc[[50175]]
print(f"Type d'attaque : {types_attaque[cible.index[0]]} | confiance de l'IDS : {confiance[cible.index[0]]:.2f}")

recettes = recettes_substitut(arbre, cible.iloc[0], list(X_test.columns))
print(f"{len(recettes)} recette(s) trouvée(s). La moins coûteuse :")
recette_en_tableau(cible, recettes[0][1])

Type d'attaque : Fuzzers | confiance de l'IDS : 0.53
2 recette(s) trouvée(s). La moins coûteuse :


,valeur actuelle,viser entre,et
feature,,,
dbytes,642.000000,642.000,873.000
dur,0.619183,0.619,3.803


**👀 Ce qu'on doit voir :** une attaque **Fuzzers** détectée de justesse (confiance 0,53). La meilleure recette ne touche que **2 features** : garder `dbytes` entre 642 et 873, et `dur` entre 0,6 et 3,8 s. Concrètement : allonger un peu la connexion.

## Étape 3 : Générer des variantes : à l'aveugle vs guidé
**Ce qu'on fait :** on tire des centaines de variantes au hasard et on garde celles que la **copie forêt** classe « normal ». Seules les **5 meilleures** sont envoyées à la vraie API.
- **À l'aveugle :** les 10 features contrôlables bougent toutes au hasard (entre leur valeur et ×3).
- **Guidé :** seules les features de la recette bougent, dans leurs plages.

In [3]:
from helper import generer_variantes, effort

for nom, guide in [("À l'aveugle", False), ('Guidé', True)]:
    variantes = generer_variantes(cible, arbre, foret, guide)
    top5 = variantes.head(5)                          # 5 requêtes seulement vers la vraie API
    passent = (api.predire(top5) == 0).sum() if len(top5) else 0
    meilleur_effort = effort(top5, cible.iloc[0])[0] if len(top5) else float('nan')
    print(f"{nom:12s} : {len(variantes)} variantes trompent la copie | effort {meilleur_effort:.2f} | passent l'IDS : {passent}/5")

À l'aveugle  : 202 variantes trompent la copie | effort 6.39 | passent l'IDS : 0/5


Guidé        : 600 variantes trompent la copie | effort 0.04 | passent l'IDS : 5/5


**👀 Ce qu'on doit voir :** **à l'aveugle** : 0/5 passent l'IDS, avec un effort de 6,4. **Guidé** : 5/5 passent, avec un effort de 0,04. La recette fait toute la différence.

In [4]:
# Avant / après : qu'est-ce qui a changé ?
from helper import CONTROLABLES

meilleure = generer_variantes(cible, arbre, foret, guide=True).head(1)
avant_apres = pd.concat([cible[CONTROLABLES], meilleure[CONTROLABLES]]).astype(float).round(3)
avant_apres.index = ['attaque originale', 'variante guidée']
print("Réponse de l'IDS sur la variante :", 'normal' if api.predire(meilleure)[0] == 0 else 'attaque')
avant_apres

Réponse de l'IDS sur la variante : normal


,sbytes,spkts,dur,smean,sinpkt,sload,rate,sjit,dbytes,dpkts
attaque originale,774.0,14.0,0.619,55.0,47.629,9289.661,40.376,3060.733,642.000,12.0
variante guidée,774.0,14.0,0.691,55.0,47.629,9289.661,40.376,3060.733,642.069,12.0


**👀 Ce qu'on doit voir :** seule la durée `dur` change vraiment : **0,619 s → 0,691 s**. Ralentir la connexion de 70 millisecondes suffit pour que l'IDS dise « normal ».

## Étape 4 : Recommencer sur 60 attaques
**Ce qu'on fait :** même attaque, mais sur 60 attaques détectées choisies au hasard, pour voir si l'exemple précédent est représentatif.

In [5]:
from helper import taux_reussite

# <-- MODIFIABLE : essayez aussi les criticismes des attaques de l'ex. 4
cibles = X_test[(y_test == 1) & (confiance > 0.5)].sample(60, random_state=0)

bilan = {nom: taux_reussite(cibles, ids, arbre, foret, guide) for nom, guide in [("à l'aveugle", False), ('guidé', True)]}
pd.DataFrame(bilan).T.round(2)

,réussite,effort médian
à l'aveugle,0.05,6.91
guidé,0.05,0.36


**👀 Ce qu'on doit voir :** les deux méthodes réussissent seulement ≈ 5 % du temps. Mais l'effort est ≈ 7 à l'aveugle contre ≈ 0,4 guidé : la copie ne rend pas l'attaque plus **probable**, elle la rend environ **20 fois plus discrète**.

**❓ Question pour la classe :** pourquoi le taux de réussite est-il si bas, même guidé ?

*Piste : l'ex. 3 a montré que l'IDS décide surtout avec `sttl`, qui n'est pas dans notre liste de features contrôlables.*

## Étape 5 : Le raccourci : et si l'attaquant touche au TTL ?
**Ce qu'on fait :** le TTL est fixé par **la machine de l'attaquant** (une option système). On prend toutes les attaques détectées et on remplace leurs valeurs de TTL par celles du trafic normal, une par une.

In [6]:
detectees = X_test[(y_test == 1) & (ids.predict(X_test) == 1)]
print('TTL le plus fréquent :', X_test.groupby(y_test)['sttl'].agg(lambda s: s.value_counts().head(3).to_dict()).to_dict(), '(0 = normal, 1 = attaque)')

# <-- MODIFIABLE : les valeurs copiées sur le trafic normal
modifs = {'sttl': 31, 'dttl': 29, 'ct_state_ttl': 0}

resultat = {}
modifiees = detectees.copy()
for feature, valeur in modifs.items():
    modifiees[feature] = valeur
    resultat[' + '.join(list(modifs)[:list(modifs).index(feature) + 1])] = (ids.predict(modifiees) == 0).mean()
pd.Series(resultat, name='attaques qui passent').round(3)

TTL le plus fréquent : {0: {31: 39455, 254: 11230, 0: 2859}, 1: {254: 103513, 62: 15514, 0: 303}} (0 = normal, 1 = attaque)


sttl                          0.093
sttl + dttl                   0.303
sttl + dttl + ct_state_ttl    0.999
Name: attaques qui passent, dtype: float64

**👀 Ce qu'on doit voir :** changer `sttl` seul fait passer ≈ 9 % des attaques, `sttl + dttl` ≈ 30 %, et les trois TTL **≈ 99,9 %**. L'IDS tient presque entièrement sur le TTL.

**❓ Question pour la classe :** l'attaquant contrôle-t-il vraiment `dttl` (le TTL de la **réponse** du serveur) et `ct_state_ttl` (un compteur calculé à partir des TTL) ? Qu'est-ce que ce résultat dit du modèle ?

## Étape 6 : Contre-mesure : retirer le raccourci (comme à l'ex. 3)
**Ce qu'on fait :** on réentraîne l'IDS sans les features TTL et on relance la même attaque.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from helper import evaluer

raccourcis = list(modifs)
ids_sans_ttl = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1).fit(X_train.drop(columns=raccourcis), y_train)
evaluer(ids_sans_ttl, X_test.drop(columns=raccourcis), y_test, 'IDS sans TTL')

print(f"Attaque TTL sur l'IDS original : {(ids.predict(modifiees) == 0).mean():.1%} passent")
print(f"Attaque TTL sur l'IDS sans TTL : {(ids_sans_ttl.predict(modifiees.drop(columns=raccourcis)) == 0).mean():.1%} passent")

IDS sans TTL -> accuracy = 0.899 | recall (attaques détectées) = 0.863


Attaque TTL sur l'IDS original : 99.9% passent


Attaque TTL sur l'IDS sans TTL : 0.5% passent


**👀 Ce qu'on doit voir :** l'IDS sans TTL est aussi bon (accuracy ≈ 0,90). Face à l'attaque TTL, **99,9 %** des attaques passent l'original, contre **0,5 %** pour l'IDS sans TTL.

## Ce qu'il faut retenir
- Copie (ex. 2) + lecture de la copie : l'attaquant remplace une recherche à l'aveugle par une recherche **ciblée** : peu de requêtes, de très petites modifications.
- Le vrai danger n'est pas la frontière fine du modèle, c'est le **raccourci** : une feature facile à falsifier qui décide presque seule.
- Un modèle sans raccourci peut être **aussi précis et beaucoup plus robuste**.

**Exercice suivant :** jusqu'ici l'attaquant attaquait l'IDS **après** son entraînement. Et s'il intervenait **avant** ?